# vlti_loader — Tutorial Notebook

This notebook walks through all features of the `vlti_loader` package for loading, inspecting, filtering, plotting, and exporting VLTI interferometric (OIFITS) data.

**Supported instruments:** GRAVITY (K-band), MATISSE (L/M/N-band), PIONIER (H-band)

In [ ]:
# ── User input ─────────────────────────────────────────────────────────────
# Set the path to your OIFITS file (or a directory containing .fits files,
# or a Python list of files from different instruments).

fits_path = ""   # ← change this

# Examples:
#   fits_path = "/data/MATISSE/mystar_LM.fits"
#   fits_path = "/data/GRAVITY/"                              # whole directory
#   fits_path = ["/data/MATISSE/lm.fits", "/data/GRAVITY/k.fits"]  # multi-instrument

## 1 — Load data

In [ ]:
import sys
sys.path.insert(0, "..")   # not needed if the package is installed via pip

from vlti_loader.VLTI_observations import Observations

obs = Observations(fits_path)
print(obs)

## 2 — Inspect with `summary()`

In [ ]:
obs.summary()

## 3 — Attribute-style data access

All keys of `obs.data` are also accessible directly as attributes.

In [ ]:
import numpy as np

# These two are equivalent:
vis2_a = obs.data['VIS2']
vis2_b = obs.VIS2
print("Arrays equal:", np.array_equal(vis2_a, vis2_b))

# Other readily available quantities:
print("Wavelength range (µm):", obs.VIS2_waves.min()*1e6, "–", obs.VIS2_waves.max()*1e6)
print("Baseline range   (m) :", obs.B.min(), "–", obs.B.max())
print("Spatial freqs (rad⁻¹):", obs.freqs.min(), "–", obs.freqs.max())
print("u / v (rad⁻¹) sample :", obs.u[:3], obs.v[:3])

# Closure phase (if available)
if 'T3_PHI' in obs.data:
    print("CP range (deg)       :", obs.T3_PHI.min(), "–", obs.T3_PHI.max())

## 4 — Angular resolution table

Prints $\lambda / B$ (in mas) per unique baseline across the observed wavelength range.

In [ ]:
obs.get_effective_resolution()

## 5 — Overview plot: UV coverage + V² + closure phase

In [ ]:
fig = obs.plot(uv_bool=True, show=True)

## 6 — Filter data

`filter_data` accepts single or multiple wavelength / baseline / frequency / error ranges.  
The operation updates `obs.data` in-place — use `reset_data()` to undo it.

In [ ]:
# Keep PIONIER (H-band), GRAVITY (K-band), and MATISSE L-band; discard very noisy points
obs.filter_data(
    wave_ranges=[(1.5e-6, 1.8e-6), (2.0e-6, 2.4e-6), (3.2e-6, 3.9e-6)],
    vis2_err_ranges=[(0, 0.15)],
    t3_err_ranges=[(0, 20)],
)
obs.summary()

In [ ]:
fig = obs.plot(uv_bool=True, v2_ylim=(0, 1.2), cp_ylim=(-30, 30))

## 7 — Spectral binning

Combine every `n` spectral channels to boost SNR at the cost of spectral resolution.  
Errors are propagated in quadrature.

In [ ]:
obs.reset_data()   # start from the full unfiltered dataset

obs.bin_spectral_channels(n=5)
obs.summary()
fig = obs.plot(uv_bool=True)

## 8 — Reset to original data

Restores `obs.data` to the original state loaded from disk — no file re-read required.

In [ ]:
obs.reset_data()
obs.summary()

## 9 — Export to OIFITS

Write the current (possibly filtered/binned) data to a valid OIFITS 2 file,
ready for model-fitting codes such as **PMOIRED**, **CANDID**, or **ASPRO2**.

> **Note:** target coordinates, telescope positions, and timestamps are not stored
> in the `vlti_loader` data model and will be written as placeholder zeros.

In [ ]:
import os

base_dir = os.path.dirname(fits_path) if not os.path.isdir(fits_path) else fits_path
output_path = os.path.join(base_dir, "filtered_output.fits")

# Apply a quality filter across all instrument bands, then export
obs.filter_data(
    wave_ranges=[(1.5e-6, 1.8e-6), (2.0e-6, 2.4e-6), (3.2e-6, 3.9e-6)],
    vis2_err_ranges=[(0, 0.15)],

)
print("Saved to:", output_path)
obs.export_oifits(output_path)